In [ ]:
%pip install -U duckduckgo-search ddgs langchain-community beautifulsoup4 # 처음 한 번만 실행

## 1. 최신 정보 한계 확인
검색 없이 물어보면 최신 정보를 모른다.

In [1]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os

load_dotenv()

model = ChatOpenAI(
    model="openai/gpt-4o-mini",                 # OpenRouter "제공자/모델" 형식
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)
model.invoke("2025년 현대자동차 미국 시장 전망은 어떻게 되나요?")

AIMessage(content='2025년 현대자동차의 미국 시장 전망은 여러 요인에 따라 달라질 수 있습니다. 다음은 몇 가지 주요 요인입니다.\n\n1. **전기차(EV) 수요 증가**: 미국 시장에서 전기차에 대한 수요가 계속해서 증가하고 있습니다. 현대자동차는 아이오닉 시리즈와 같은 전기차 라인업을 강화하고 있어 이 부분에서 경쟁력을 가질 가능성이 높습니다.\n\n2. **정부의 친환경 정책**: 미국 정부의 친환경 정책 및 전기차 관련 보조금이 지속된다면, 이는 현대자동차의 판매에 긍정적인 영향을 미칠 수 있습니다.\n\n3. **경쟁 환경**: 테슬라, 포드, GM 등과 같은 다른 자동차 제조사들과의 경쟁이 치열할 것으로 예상됩니다. 현대자동차가 이들 사이에서 시장 점유율을 유지하거나 확대하기 위해서는 혁신적인 모델 및 가격 경쟁력을 확보해야 할 것입니다.\n\n4. **소비자 선호 변화**: 소비자들의 자동차 선호가 전통적인 내연기관 차량에서 전기차 혹은 하이브리드 차량으로 이동하고 있는 추세가 계속될 경우, 현대자동차의 EV 관련 제품군의 성장이 더욱 중요할 것입니다.\n\n5. **글로벌 공급망**: 반도체 및 부품 부족과 같은 공급망 문제는 여전히 영향을 미칠 수 있습니다. 이러한 문제가 해결된다면 생산 능력이 개선되어 더 많은 차량을 시장에 공급할 수 있을 것입니다.\n\n결론적으로, 현대자동차는 전기차 중심의 전략과 함께 시장 변화에 유연하게 대응한다면 2025년 미국 시장에서 긍정적인 성과를 거둘 가능성이 있습니다. 그러나 경쟁과 글로벌 경제 상황 등 다양한 변수에 의해 전망은 달라질 수 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 388, 'prompt_tokens': 21, 'total_tokens': 409, 'completion_tokens_details': {'accepted_predictio

## 2. 덕덕고 검색 테스트

In [2]:
from langchain_community.tools import DuckDuckGoSearchResults

search = DuckDuckGoSearchResults(results_separator=";\n")
docs = search.invoke("2025년 현대자동차 미국 시장 전망은 어떻게 되나요?")

print(docs)

C:\Users\kim10\AppData\Local\Temp\ipykernel_26116\715596964.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchResults


snippet: 2025. 7. 23. · 2025년 글로벌 신차시장은 당초 전년도 2%대의 낮은 성장률에 따른 기저효과와 금리 인하, 공급망 안정화로 인한 생산증가 등으로 전년 대비 약 3% 성장이 예상 ..., title: 2025년 자동차산업 상반기 동향 및 하반기 전망 > 기고문, link: https://kaica.or.kr/contribute/1379?sst=wr_datetime;
snippet: 2026. 1. 28. · -2025년 연간 매출 186조 2,545억 원 : 전년 대비 6.3% 증가로 사상 최대 매출액 ㄷㄷ → 지난해 미국 자동차 관세 여파로 인해 역대 최대 매출에도 영업이익이 ..., title: 매출 사상 최대 기록한 현대차 실적 발표 -2025년 연간 영업이익 11조 ..., link: https://www.instagram.com/p/DUFUJDwEgP1/;
snippet: 2026. 5. 22. · Q1. 2026년 글로벌 자동차 시장 전망은 어떻게 되나요? 지역별 양극화가 심화되고 있습니다. 미국은 고금리·고유가 속 HEV 중심 방어적 수요, 유럽은 BEV·HEV 동반 ..., title: 2026 하반기 자동차 산업 전망: 마력보다 알고리즘이 중요 - 신한금융그룹, link: https://www.shinhangroup.com/kr/archive/insight/extend/detail/32863;
snippet: 2026. 1. 19. · 2025년 미국 800V 전기차 아키텍처 시장 규모는 11.3억 달러로 평가되었습니다. 이 시장은 2026년 13.4억 달러에서 2035년 99.2억 달러로 성장할 것으로 예상되며, ..., title: 미국 800V 전기차 아키텍처 시장 규모, 2035년 보고서, link: https://www.gminsights.com/ko/industry-analysis/us-800v-electric-vehicle-architecture-market


## 3. LCEL 방식으로 프롬프트 + GPT 모델 연결

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

question_answering_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "사용자의 질문에 대해 아래 context에 기반하여 답변하라.:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

document_chain = question_answering_prompt | model

## 4. 메시지 저장하고 답변 생성

In [ ]:
from langchain.memory import ChatMessageHistory

# 채팅 메시지를 저장할 메모리 객체 생성
chat_history = ChatMessageHistory()
# 사용자 질문을 메모리에 저장
chat_history.add_user_message("2025년 현대자동차 미국 시장 전망은 어떻게 되나요?")

# 문서 검색하고 답변 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": docs,
    }
)

# 생성된 답변을 메모리에 저장
chat_history.add_ai_message(answer.content)

print(answer.content)

## 5. API 래퍼(Wrapper)로 검색 옵션 설정
한국 지역, 최근 1주일, 뉴스 소스 위주로 검색.

In [ ]:
# DuckDuckGo API wrapper 로 검색 매개변수를 설정
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 한국 지역("kr-kr"), 최근 일주일("w") 내의 검색 결과를 가져오도록 초기화
wrapper = DuckDuckGoSearchAPIWrapper(region="kr-kr", time="w")

# 뉴스 소스 위주로 검색하도록 지정
search = DuckDuckGoSearchResults(
    api_wrapper=wrapper,
    source="news",
    results_separator=";\n",
)

docs = search.invoke("2025년 현대자동차 미국 시장 전망은 어떻게 되나요?")
print(docs)

## 6. 특정 웹 사이트만 검색
`site:` 연산자 사용.

In [ ]:
docs = search.invoke("site:ytn.co.kr 2025년 현대자동차 미국 시장 전망은 어떻게 되나요?")
docs

## 7. 검색 결과에서 링크(URL) 추출

In [ ]:
# 검색 결과의 링크들을 저장할 빈 리스트 초기화
links = []

# 검색 결과를 세미콜론과 줄바꿈 기준으로 분리하고, 각 항목에서 링크를 추출
for doc in docs.split(";\n"):
    print(doc)  # 각 검색 결과 항목 확인
    link = doc.split("link:")[1].strip()  # 'link:' 이후의 URL 부분만 추출
    links.append(link)

print(links)

## 8. 웹 페이지 내용을 비동기로 읽어오기
노트북은 top-level await 를 지원하므로 `async for` 를 바로 쓸 수 있다.

In [ ]:
# WebBaseLoader 로 웹 페이지 내용을 불러온다.
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=links,                  # 웹 페이지 링크 목록
    bs_get_text_kwargs={"strip": True},  # 앞뒤 공백 제거
)

page_contents = []
async for doc in loader.alazy_load():
    page_contents.append(doc)

for content in page_contents:
    print(content)
    print('--------------')

## 9. BeautifulSoup 으로 기사 본문만 추출

In [ ]:
import requests
from bs4 import BeautifulSoup

# 주어진 URL에서 기사 텍스트를 가져오는 함수
def get_article_text(url):
    try:
        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "html.parser")

        # 클래스가 'story-news article'인 <article> 태그를 찾음
        article = soup.find('article', class_='story-news article')

        if article:
            return article.get_text(strip=True)
        else:
            try:
                if soup.find('article'):
                    return soup.find('article').get_text(strip=True)
                elif soup.find("div", id="CmAdContent"):
                    return soup.find("div", id="CmAdContent").get_text(strip=True)
            except:
                return "기사 내용을 찾을 수 없습니다."
    except requests.exceptions.RequestException as e:
        return f"URL을 가져오는 중 오류 발생: {e}"

In [ ]:
# 각 링크를 반복하면서 기사 텍스트를 출력
articles = []
for link in links:
    print(f"URL: {link}\n")
    article_text = get_article_text(link)
    print(f"Content:\n{article_text}")
    print("--------------------------------------------------")
    articles.append(article_text)

## 10. 검색 결과를 바탕으로 최종 답변 생성

In [ ]:
chat_history.add_user_message("2025년 현대자동차 미국 시장 전망은 어떻게 되나요?")

# 문서 검색하고 답변을 생성
answer = document_chain.invoke(
    {
        "messages": chat_history.messages,
        "context": articles,
    }
)

# 생성된 답변 메모리에 저장
chat_history.add_ai_message(answer.content)
print(answer.content)